# AMD Hackathon — Vulnerability-Aware Java LLM

Script 2: Clean datasets, apply unified schema, export to JSONL (chat-template)

Unified Schema (per row):
  id                  — unique hash-based ID
  source              — which original dataset
  cve_id              — CVE-YYYY-NNNNN  (nullable)
  cwe_id              — CWE-NNN         (nullable)
  cwe_name            — Human-readable CWE name (nullable)
  vulnerability_type  — Normalised type string (e.g. "SQL Injection", "CWE-079")
  vulnerable_code     — Java code snippet BEFORE the fix
  fixed_code          — Java code snippet AFTER the fix  (empty if classification-only)
  diff                — Unified diff / patch            (nullable)
  explanation         — Narrative description of the fix (nullable)
  language            — always "java"
  repo_url            — GitHub repo URL                 (nullable)
  commit_hash         — fix commit SHA                  (nullable)
  file_path           — path of the changed file        (nullable)
  method_name         — changed method name             (nullable)
  has_fix             — bool: whether fixed_code is present

Output:
  output/unified_dataset.jsonl      ← flat unified records
  output/train_chat_template.jsonl  ← Unsloth / SFT chat-template format
  output/unified_dataset.csv        ← optional CSV export
  output/stats.json                 ← dataset statistics

Usage:
  pip install pandas tqdm
  python 02_clean_and_convert.py

  Optional flags:
    --skip-barryallen16   skip classification-only dataset (no fixed_code)
    --min-code-len 50     minimum char length for vulnerable_code (default 50)
    --max-code-len 8000   maximum char length for vulnerable_code (default 8000)
    --no-csv              don't write CSV output

In [ ]:
from __future__ import annotations

import csv
import hashlib
import json
import os
import re
import sys
import argparse
import sqlite3
from pathlib import Path
from typing import Iterator, Optional

RAW_DIR   = Path("raw_data")
OUT_DIR   = Path("output")

# ─── CWE name lookup (most common in Java codebases) ─────────────────────────
CWE_NAMES: dict[str, str] = {
    "CWE-022": "Path Traversal",
    "CWE-078": "OS Command Injection",
    "CWE-079": "Cross-Site Scripting (XSS)",
    "CWE-089": "SQL Injection",
    "CWE-090": "LDAP Injection",
    "CWE-094": "Code Injection",
    "CWE-095": "Eval Injection",
    "CWE-113": "HTTP Response Splitting",
    "CWE-116": "Improper Encoding",
    "CWE-134": "Uncontrolled Format String",
    "CWE-190": "Integer Overflow",
    "CWE-200": "Exposure of Sensitive Information",
    "CWE-287": "Improper Authentication",
    "CWE-295": "Improper Certificate Validation",
    "CWE-306": "Missing Authentication",
    "CWE-311": "Missing Encryption",
    "CWE-326": "Inadequate Encryption Strength",
    "CWE-327": "Use of Broken Algorithm",
    "CWE-330": "Use of Insufficiently Random Values",
    "CWE-352": "Cross-Site Request Forgery (CSRF)",
    "CWE-400": "Uncontrolled Resource Consumption",
    "CWE-434": "Unrestricted File Upload",
    "CWE-476": "NULL Pointer Dereference",
    "CWE-502": "Deserialization of Untrusted Data",
    "CWE-601": "URL Redirection to Untrusted Site",
    "CWE-611": "XML External Entity (XXE)",
    "CWE-639": "Insecure Direct Object Reference",
    "CWE-732": "Incorrect Permission Assignment",
    "CWE-787": "Out-of-bounds Write",
    "CWE-918": "Server-Side Request Forgery (SSRF)",
}

# Mapping from informal names found in barryallen16 → normalised vulnerability_type
VULN_TYPE_NORMALISE: dict[str, str] = {
    "sql injection":                  "SQL Injection",
    "sqli":                           "SQL Injection",
    "command injection":              "OS Command Injection",
    "os command injection":           "OS Command Injection",
    "rce":                            "Remote Code Execution",
    "remote code execution":          "Remote Code Execution",
    "xss":                            "Cross-Site Scripting (XSS)",
    "cross-site scripting":           "Cross-Site Scripting (XSS)",
    "xxe":                            "XML External Entity (XXE)",
    "xml external entity":            "XML External Entity (XXE)",
    "path traversal":                 "Path Traversal",
    "directory traversal":            "Path Traversal",
    "buffer overflow":                "Buffer Overflow",
    "csrf":                           "Cross-Site Request Forgery (CSRF)",
    "cross-site request forgery":     "Cross-Site Request Forgery (CSRF)",
    "ssrf":                           "Server-Side Request Forgery (SSRF)",
    "ldap injection":                 "LDAP Injection",
    "code injection":                 "Code Injection",
    "deserialization":                "Insecure Deserialization",
    "insecure deserialization":       "Insecure Deserialization",
    "open redirect":                  "Open Redirect",
    "information disclosure":         "Information Exposure",
    "improper authentication":        "Improper Authentication",
    "broken authentication":          "Improper Authentication",
    "safe":                           "NOT_VULNERABLE",
    "not_vulnerable":                 "NOT_VULNERABLE",
}

## Helpers

In [ ]:
def make_id(source: str, content: str) -> str:
    """Deterministic short ID based on source + content hash."""
    return source[:6] + "_" + hashlib.md5(content.encode()).hexdigest()[:12]

In [ ]:
def normalise_vuln_type(raw: str) -> str:
    """Map informal vulnerability labels to a standardised form."""
    if not raw:
        return "Unknown"
    key = raw.strip().lower()
    return VULN_TYPE_NORMALISE.get(key, raw.strip())

In [ ]:
def clean_java_code(code: str) -> str:
    """
    Basic clean-up for Java code snippets:
      - Remove leading/trailing whitespace
      - Normalise internal indentation (strip excess common indent)
    """
    if not code:
        return ""
    lines = code.splitlines()
    # Strip blank-only prefix/suffix lines
    while lines and not lines[0].strip():
        lines.pop(0)
    while lines and not lines[-1].strip():
        lines.pop()
    if not lines:
        return ""
    # Detect and strip common leading whitespace
    non_empty = [l for l in lines if l.strip()]
    if non_empty:
        min_indent = min(len(l) - len(l.lstrip()) for l in non_empty)
        lines = [l[min_indent:] if len(l) >= min_indent else l for l in lines]
    return "\n".join(lines)

In [ ]:
def is_java_code(code: str) -> bool:
    """Heuristic check that the string looks like Java source code."""
    if not code or len(code) < 20:
        return False
    java_hints = [
        r'\bclass\b', r'\bpublic\b', r'\bprivate\b', r'\bprotected\b',
        r'\bvoid\b', r'\breturn\b', r'\bimport\b', r'\bpackage\b',
        r'\bthrows\b', r'\bextends\b', r'\bimplements\b',
    ]
    hits = sum(1 for p in java_hints if re.search(p, code))
    return hits >= 2

In [ ]:
def make_record(
    source: str,
    vulnerable_code: str,
    fixed_code: str = "",
    diff: str = "",
    vulnerability_type: str = "Unknown",
    cve_id: str = "",
    cwe_id: str = "",
    cwe_name: str = "",
    explanation: str = "",
    repo_url: str = "",
    commit_hash: str = "",
    file_path: str = "",
    method_name: str = "",
) -> dict:
    """Build a unified schema record."""
    vuln_code_clean = clean_java_code(vulnerable_code)
    fixed_code_clean = clean_java_code(fixed_code)
    cwe_id_norm = cwe_id.upper().strip() if cwe_id else ""
    if cwe_id_norm and not cwe_id_norm.startswith("CWE-"):
        cwe_id_norm = f"CWE-{cwe_id_norm.lstrip('0')}"
    # Fill cwe_name from lookup if missing
    if cwe_id_norm and not cwe_name:
        cwe_name = CWE_NAMES.get(cwe_id_norm, "")

    return {
        "id":                 make_id(source, vuln_code_clean + fixed_code_clean + cve_id),
        "source":             source,
        "cve_id":             cve_id.strip() if cve_id else "",
        "cwe_id":             cwe_id_norm,
        "cwe_name":           cwe_name.strip() if cwe_name else "",
        "vulnerability_type": normalise_vuln_type(vulnerability_type) if vulnerability_type else (
                                  cwe_name or cwe_id_norm or "Unknown"),
        "vulnerable_code":    vuln_code_clean,
        "fixed_code":         fixed_code_clean,
        "diff":               diff.strip() if diff else "",
        "explanation":        explanation.strip() if explanation else "",
        "language":           "java",
        "repo_url":           repo_url.strip() if repo_url else "",
        "commit_hash":        commit_hash.strip() if commit_hash else "",
        "file_path":          file_path.strip() if file_path else "",
        "method_name":        method_name.strip() if method_name else "",
        "has_fix":            bool(fixed_code_clean),
    }

## Parser: Dataset 1 — barryallen16

In [ ]:
def parse_barryallen16(skip: bool = False) -> Iterator[dict]:
    """
    Parse barryallen16/java_vulnerability_dataset.
    Only yields rows where the code is VULNERABLE (no fixed_code available).
    Useful for teaching the model to recognize vulnerability patterns.
    """
    if skip:
        print("  [SKIP] barryallen16 (--skip-barryallen16 flag)")
        return

    src = RAW_DIR / "barryallen16"
    csv_path = src / "java_vul_dataset.csv"
    if not csv_path.exists():
        print(f"  [WARN] {csv_path} not found. Run 01_download_datasets.py first.")
        return

    count = 0
    with open(csv_path, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            code     = row.get("code", "").strip()
            expected = row.get("expected", "").strip()

            if not code:
                continue

            # Parse "Classification: VULNERABLE. Vulnerability Type: SQL Injection"
            vuln_match   = re.search(r'Classification:\s*(VULNERABLE|NOT_VULNERABLE)', expected, re.I)
            vtype_match  = re.search(r'Vulnerability Type:\s*(.+?)(?:\.|$)', expected, re.I)

            classification = vuln_match.group(1).upper() if vuln_match else "UNKNOWN"
            vtype          = vtype_match.group(1).strip() if vtype_match else "Unknown"

            # Only include VULNERABLE samples (no fixed_code for this dataset)
            if classification != "VULNERABLE":
                continue

            if not is_java_code(code):
                continue

            rec = make_record(
                source="barryallen16",
                vulnerable_code=code,
                fixed_code="",        # classification-only dataset
                vulnerability_type=vtype,
            )
            yield rec
            count += 1

    print(f"  barryallen16: yielded {count} VULNERABLE rows from java_vul_dataset.csv")

    # Also parse finetuning_dataset_v4.jsonl if available (richest version)
    jsonl_path = src / "finetuning_dataset_v4.jsonl"
    if not jsonl_path.exists():
        jsonl_path = src / "finetuning_dataset_v3.jsonl"
    if not jsonl_path.exists():
        jsonl_path = src / "finetuning_dataset.jsonl"

    if jsonl_path.exists():
        count2 = 0
        with open(jsonl_path, encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    obj = json.loads(line)
                except json.JSONDecodeError:
                    continue

                # Format: list of {role, content}
                messages = obj if isinstance(obj, list) else obj.get("messages", [])
                if not messages:
                    continue

                # Extract user message content
                user_content = next(
                    (m["content"] for m in messages if m.get("role") == "user"), ""
                )
                assistant_content = next(
                    (m["content"] for m in messages if m.get("role") == "assistant"), ""
                )

                # User message may contain JSON with 'code' field
                code = ""
                vtype = "Unknown"
                try:
                    # content might be a JSON string itself
                    inner = json.loads(user_content) if user_content.startswith("{") else {}
                    code = inner.get("code", "")
                except Exception:
                    # Fall back to extracting code block
                    m = re.search(r'```(?:java)?\n([\s\S]+?)```', user_content)
                    if m:
                        code = m.group(1)

                # Extract classification from assistant message
                vuln_match  = re.search(r'Classification:\s*(VULNERABLE|NOT_VULNERABLE)', assistant_content, re.I)
                vtype_match = re.search(r'Vulnerability Type:\s*(.+?)(?:\.|$)', assistant_content, re.I)

                classification = vuln_match.group(1).upper() if vuln_match else "UNKNOWN"
                vtype = vtype_match.group(1).strip() if vtype_match else "Unknown"

                if classification != "VULNERABLE" or not code:
                    continue
                if not is_java_code(code):
                    continue

                rec = make_record(
                    source="barryallen16",
                    vulnerable_code=code,
                    fixed_code="",
                    vulnerability_type=vtype,
                )
                yield rec
                count2 += 1

        print(f"  barryallen16: yielded {count2} VULNERABLE rows from {jsonl_path.name}")

## Parser: Dataset 2 — JavaVFC

In [ ]:
def parse_javavfc() -> Iterator[dict]:
    """
    Parse JavaVFC javavfc.jsonl from Zenodo.
    Expected JSONL fields (based on similar VFC datasets):
      commit_id / hash, repo_url / project, message,
      diff / patch, code_before, code_after, cve_id, cwe_id
    We handle both possible column naming conventions.
    """
    src = RAW_DIR / "javavfc"
    jsonl_path = src / "javavfc.jsonl"
    if not jsonl_path.exists():
        print(f"  [WARN] {jsonl_path} not found. Run 01_download_datasets.py first.")
        return

    count = skip_count = 0
    with open(jsonl_path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
            except json.JSONDecodeError:
                continue

            # ── Flexible field extraction ──────────────────────────────────
            # Common naming variants across VFC/CVEfixes-style datasets
            vulnerable_code = (
                obj.get("code_before") or obj.get("before") or
                obj.get("vulnerable_code") or obj.get("buggy_code") or ""
            )
            fixed_code = (
                obj.get("code_after") or obj.get("after") or
                obj.get("fixed_code") or obj.get("patched_code") or ""
            )
            diff = (
                obj.get("diff") or obj.get("patch") or obj.get("changes") or ""
            )
            commit_hash = (
                obj.get("commit_id") or obj.get("hash") or obj.get("commit") or ""
            )
            repo_url = (
                obj.get("repo_url") or obj.get("project") or
                obj.get("repository") or obj.get("repo") or ""
            )
            cve_id = obj.get("cve_id") or obj.get("cve") or ""
            cwe_id = obj.get("cwe_id") or obj.get("cwe") or ""
            message = obj.get("message") or obj.get("commit_message") or obj.get("msg") or ""

            # If code_before/code_after are absent but diff is present, try to extract them
            if not vulnerable_code and diff:
                removed = [l[1:] for l in diff.splitlines() if l.startswith("-") and not l.startswith("---")]
                if removed:
                    vulnerable_code = "\n".join(removed)
            if not fixed_code and diff:
                added = [l[1:] for l in diff.splitlines() if l.startswith("+") and not l.startswith("+++")]
                if added:
                    fixed_code = "\n".join(added)

            # Skip rows without at least vulnerable code
            if not vulnerable_code:
                skip_count += 1
                continue

            # Filter to Java if language info available
            lang = (obj.get("language") or obj.get("programming_language") or "").lower()
            if lang and "java" not in lang:
                skip_count += 1
                continue

            if not is_java_code(vulnerable_code) and not is_java_code(fixed_code):
                skip_count += 1
                continue

            rec = make_record(
                source="javavfc",
                vulnerable_code=vulnerable_code,
                fixed_code=fixed_code,
                diff=diff,
                cve_id=cve_id,
                cwe_id=cwe_id,
                explanation=message,
                repo_url=repo_url,
                commit_hash=commit_hash,
            )
            yield rec
            count += 1

    print(f"  JavaVFC: yielded {count} rows (skipped {skip_count})")

## Parser: Dataset 3 — CWE-Bench-Java

In [ ]:
def parse_cwe_bench_java() -> Iterator[dict]:
    """
    Parse CWE-Bench-Java metadata CSVs.

    These CSVs contain CVE/CWE metadata + GitHub references but NOT source code.
    We yield metadata-only records tagged as has_fix=False with an explanation
    of what commit contains the fix. These enrich the other datasets with
    CVE/CWE metadata during merging.

    For actual code, use 01b_fetch_cwe_bench_code.py which calls the GitHub API.
    """
    src = RAW_DIR / "cwe_bench_java"
    project_csv = src / "project_info.csv"
    fix_csv     = src / "fix_info.csv"

    if not project_csv.exists():
        print(f"  [WARN] {project_csv} not found. Run 01_download_datasets.py first.")
        return

    # Build CWE lookup from project_info
    cve_to_meta: dict[str, dict] = {}
    with open(project_csv, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            cve_id = row.get("cve_id", "").strip()
            if cve_id:
                cve_to_meta[cve_id] = {
                    "cwe_id":   row.get("cwe_id", ""),
                    "cwe_name": row.get("cwe_name", ""),
                    "repo_url": row.get("github_url", ""),
                    "fix_commits": row.get("fix_commit_ids", "").split(";"),
                }

    # Check for code files (fetched separately by 01b_fetch_cwe_bench_code.py)
    code_dir = src / "code"
    if code_dir.exists():
        print(f"  CWE-Bench-Java: found code dir {code_dir}, loading fetched code...")
        count = 0
        for code_file in code_dir.glob("*_before.java"):
            stem = code_file.stem.replace("_before", "")
            after_file = code_dir / f"{stem}_after.java"
            meta_file  = code_dir / f"{stem}_meta.json"

            vulnerable_code = code_file.read_text(encoding="utf-8", errors="ignore")
            fixed_code = after_file.read_text(encoding="utf-8", errors="ignore") if after_file.exists() else ""
            meta = json.loads(meta_file.read_text()) if meta_file.exists() else {}

            cve_id   = meta.get("cve_id", "")
            cwe_info = cve_to_meta.get(cve_id, {})

            rec = make_record(
                source="cwe_bench_java",
                vulnerable_code=vulnerable_code,
                fixed_code=fixed_code,
                cve_id=cve_id,
                cwe_id=cwe_info.get("cwe_id", ""),
                cwe_name=cwe_info.get("cwe_name", ""),
                repo_url=cwe_info.get("repo_url", ""),
                commit_hash=meta.get("commit_hash", ""),
                file_path=meta.get("file_path", ""),
                method_name=meta.get("method_name", ""),
            )
            yield rec
            count += 1
        print(f"  CWE-Bench-Java: yielded {count} rows from fetched code.")
    else:
        print(
            "  [INFO] CWE-Bench-Java: No code fetched yet.\n"
            "         Run python 01b_fetch_cwe_bench_code.py to pull code from GitHub.\n"
            f"         Metadata loaded: {len(cve_to_meta)} CVE entries in lookup table."
        )
        # Yield nothing — we only include rows with actual code

## Parser: Dataset 4 — CVEFixes

In [ ]:
def parse_cvefixes() -> Iterator[dict]:
    """
    Parse the CVEFixes dataset.

    Supports two formats:
      A. Kaggle CSV export  (raw_data/cvefixes/*.csv)
      B. Original SQLite DB (raw_data/cvefixes/CVEfixes.db)
    """
    src = RAW_DIR / "cvefixes"

    # ── Option A: CSV (Kaggle girish17019 export) ────────────────────────────
    csv_candidates = list(src.glob("*.csv"))
    if csv_candidates:
        yield from _parse_cvefixes_csv(src, csv_candidates)
        return

    # ── Option B: SQLite DB ──────────────────────────────────────────────────
    db_path = src / "CVEfixes.db"
    if db_path.exists():
        yield from _parse_cvefixes_sqlite(db_path)
        return

    print(
        f"  [WARN] No CVEFixes data found in {src}.\n"
        "         See raw_data/cvefixes/DOWNLOAD_INSTRUCTIONS.txt"
    )

In [ ]:
def _parse_cvefixes_csv(src: Path, csv_files: list) -> Iterator[dict]:
    """Parse Kaggle CSV export of CVEFixes (Java filtered)."""
    # The Kaggle export typically has columns from file_changes + method_changes joined
    # Common column names: cve_id, code_before, code_after, diff, programming_language,
    #                      filename, hash/commit_hash, repo_url, cwe_id, message
    count = skip_count = 0

    for csv_path in csv_files:
        print(f"  CVEFixes: parsing {csv_path.name} ...")
        with open(csv_path, newline="", encoding="utf-8", errors="ignore") as f:
            # Sniff delimiter
            sample = f.read(4096)
            f.seek(0)
            try:
                dialect = csv.Sniffer().sniff(sample)
            except csv.Error:
                dialect = csv.excel

            reader = csv.DictReader(f, dialect=dialect)
            for row in reader:
                lang = (row.get("programming_language") or row.get("language") or "").lower()
                if lang and "java" not in lang:
                    skip_count += 1
                    continue

                vulnerable_code = row.get("code_before") or row.get("before") or ""
                fixed_code      = row.get("code_after")  or row.get("after")  or ""
                diff            = row.get("diff") or ""
                cve_id          = row.get("cve_id") or ""
                cwe_id          = row.get("cwe_id") or row.get("cwe") or ""
                commit_hash     = row.get("hash") or row.get("commit_hash") or row.get("commit") or ""
                repo_url        = row.get("repo_url") or row.get("repository") or ""
                file_path       = row.get("filename") or row.get("file") or row.get("filepath") or ""
                method_name     = row.get("name") or row.get("method") or row.get("method_name") or ""
                message         = row.get("msg") or row.get("message") or row.get("commit_message") or ""

                if not vulnerable_code and not diff:
                    skip_count += 1
                    continue
                if not is_java_code(vulnerable_code) and not is_java_code(fixed_code):
                    skip_count += 1
                    continue

                rec = make_record(
                    source="cvefixes",
                    vulnerable_code=vulnerable_code,
                    fixed_code=fixed_code,
                    diff=diff,
                    cve_id=cve_id,
                    cwe_id=cwe_id,
                    explanation=message,
                    repo_url=repo_url,
                    commit_hash=commit_hash,
                    file_path=file_path,
                    method_name=method_name,
                )
                yield rec
                count += 1

    print(f"  CVEFixes (CSV): yielded {count} rows (skipped {skip_count})")

In [ ]:
def _parse_cvefixes_sqlite(db_path: Path) -> Iterator[dict]:
    """
    Parse the original CVEFixes SQLite database, filtering for Java.
    Joins: fixes → commits → file_changes → method_changes
    """
    print(f"  CVEFixes: connecting to SQLite: {db_path}")
    conn = sqlite3.connect(db_path)
    conn.row_factory = sqlite3.Row
    cur = conn.cursor()

    query = """
        SELECT
            f.cve_id,
            c.repo_url,
            c.hash        AS commit_hash,
            c.msg         AS message,
            fc.filename,
            fc.diff,
            fc.code_before,
            fc.code_after,
            fc.programming_language,
            cv.cwe        AS cwe_id
        FROM fixes f
        JOIN commits c      ON f.hash = c.hash AND f.repo_url = c.repo_url
        JOIN file_changes fc ON c.hash = fc.hash
        LEFT JOIN cve cv    ON f.cve_id = cv.cve_id
        WHERE LOWER(fc.programming_language) = 'java'
          AND fc.code_before IS NOT NULL
          AND fc.code_before != ''
    """

    count = 0
    try:
        cur.execute(query)
    except sqlite3.OperationalError:
        # Fall back to simpler query if schema differs
        query_simple = """
            SELECT fc.*, f.cve_id, c.repo_url, c.hash AS commit_hash, c.msg AS message
            FROM file_changes fc
            LEFT JOIN commits c ON fc.hash = c.hash
            LEFT JOIN fixes f ON c.hash = f.hash
            WHERE LOWER(fc.programming_language) = 'java'
              AND fc.code_before IS NOT NULL
              AND fc.code_before != ''
        """
        cur.execute(query_simple)

    for row in cur:
        row = dict(row)
        rec = make_record(
            source="cvefixes",
            vulnerable_code=row.get("code_before", ""),
            fixed_code=row.get("code_after", ""),
            diff=row.get("diff", ""),
            cve_id=row.get("cve_id", ""),
            cwe_id=row.get("cwe_id", ""),
            explanation=row.get("message", ""),
            repo_url=row.get("repo_url", ""),
            commit_hash=row.get("commit_hash", ""),
            file_path=row.get("filename", ""),
        )
        yield rec
        count += 1

    conn.close()
    print(f"  CVEFixes (SQLite): yielded {count} Java rows")

## Chat template builder  (Unsloth / SFT JSONL format)

In [ ]:
SYSTEM_PROMPT = (
    "You are an expert Java security engineer specialising in vulnerability remediation. "
    "When given a security vulnerability report and vulnerable Java code, your task is to "
    "produce a corrected, secure version of the code that eliminates the vulnerability "
    "while preserving the original logic and functionality. "
    "Output ONLY the fixed Java code inside a ```java ... ``` block. "
    "Do not include explanations unless explicitly requested."
)

SYSTEM_PROMPT_WITH_EXPLANATION = (
    "You are an expert Java security engineer specialising in vulnerability remediation. "
    "When given a security vulnerability report and vulnerable Java code, your task is to:\n"
    "1. Briefly explain the root cause of the vulnerability.\n"
    "2. Provide the corrected, secure Java code inside a ```java ... ``` block.\n"
    "Preserve the original program logic and only change what is necessary to fix the issue."
)

def build_user_prompt(record: dict) -> str:
    """Build the user turn from a unified record."""
    lines = ["## Security Vulnerability Report\n"]

    if record["cve_id"]:
        lines.append(f"**CVE ID:** {record['cve_id']}")
    if record["cwe_id"]:
        label = record["cwe_name"] or record["cwe_id"]
        lines.append(f"**Weakness:** {record['cwe_id']} — {label}")
    if record["vulnerability_type"] and record["vulnerability_type"] not in ("Unknown", "NOT_VULNERABLE"):
        lines.append(f"**Vulnerability Type:** {record['vulnerability_type']}")
    if record["repo_url"]:
        lines.append(f"**Affected Repository:** {record['repo_url']}")
    if record["file_path"]:
        lines.append(f"**File:** `{record['file_path']}`")
    if record["method_name"]:
        lines.append(f"**Method:** `{record['method_name']}`")

    lines.append("\n## Vulnerable Java Code\n")
    lines.append(f"```java\n{record['vulnerable_code']}\n```\n")
    lines.append("Please provide the fixed, secure version of this code.")

    return "\n".join(lines)

In [ ]:
def build_assistant_response(record: dict, include_explanation: bool = False) -> str:
    """Build the assistant turn from a unified record."""
    fixed = record.get("fixed_code", "")
    if not fixed:
        return ""  # Skip if no fix available

    parts = []

    if include_explanation and record.get("explanation"):
        parts.append(f"**Root Cause:** {record['explanation']}\n")

    parts.append(f"```java\n{fixed}\n```")
    return "\n".join(parts)

In [ ]:
def to_chat_template(record: dict, include_explanation: bool = False) -> Optional[dict]:
    """
    Convert a unified record to Unsloth / SFT chat-template format.
    Returns None if the record has no fixed_code (can't train fix generation).
    """
    if not record.get("fixed_code"):
        return None  # Skip classification-only rows for chat template

    assistant = build_assistant_response(record, include_explanation)
    if not assistant:
        return None

    sys_prompt = SYSTEM_PROMPT_WITH_EXPLANATION if include_explanation else SYSTEM_PROMPT

    return {
        "messages": [
            {"role": "system",    "content": sys_prompt},
            {"role": "user",      "content": build_user_prompt(record)},
            {"role": "assistant", "content": assistant},
        ]
    }

## Quality filters

In [ ]:
def quality_filter(record: dict, min_code_len: int = 50, max_code_len: int = 8000) -> bool:
    """Return True if the record passes quality checks."""
    vcode = record["vulnerable_code"]

    # Length checks on vulnerable_code
    if len(vcode) < min_code_len:
        return False
    if len(vcode) > max_code_len:
        return False

    # Must look like Java
    if not is_java_code(vcode):
        return False

    # If has_fix, fixed_code must also be non-trivial
    if record["has_fix"]:
        fcode = record["fixed_code"]
        if len(fcode) < min_code_len // 2:
            return False
        # Reject if fixed == vulnerable (no actual change)
        if vcode.strip() == fcode.strip():
            return False

    # Skip pure "safe" / NOT_VULNERABLE rows (they shouldn't appear, but just in case)
    if record["vulnerability_type"] == "NOT_VULNERABLE":
        return False

    return True

In [ ]:
def deduplicate(records: list[dict]) -> list[dict]:
    """Remove exact duplicates by (vulnerable_code, fixed_code) pair."""
    seen: set[str] = set()
    out = []
    for r in records:
        key = hashlib.md5((r["vulnerable_code"] + r["fixed_code"]).encode()).hexdigest()
        if key not in seen:
            seen.add(key)
            out.append(r)
    return out

## Main pipeline

In [ ]:
# ── Configuration — edit these variables before running ───────────────────────

# Set True to skip the classification-only barryallen16 dataset (no fixed_code)
SKIP_BARRYALLEN16 = False

# Code length filter for vulnerable_code snippets
MIN_CODE_LEN = 50    # minimum character length
MAX_CODE_LEN = 8000  # maximum character length

# Set False to skip CSV export (saves disk space)
WRITE_CSV = True

# Set True to add a root-cause explanation turn in the chat template
INCLUDE_EXPLANATION = False

In [ ]:
# ── Run pipeline ───────────────────────────────────────────────────────────────
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("\n=== Parsing datasets ===")
all_records = []
for rec in parse_barryallen16(skip=SKIP_BARRYALLEN16):
    all_records.append(rec)
for rec in parse_javavfc():
    all_records.append(rec)
for rec in parse_cwe_bench_java():
    all_records.append(rec)
for rec in parse_cvefixes():
    all_records.append(rec)

print(f"\nRaw collected records: {len(all_records)}")

print("\n=== Quality filtering ===")
filtered = [r for r in all_records if quality_filter(r, MIN_CODE_LEN, MAX_CODE_LEN)]
print(f"After quality filter: {len(filtered)} (removed {len(all_records) - len(filtered)})")

print("\n=== Deduplication ===")
deduped = deduplicate(filtered)
print(f"After dedup: {len(deduped)} (removed {len(filtered) - len(deduped)} duplicates)")

sources, vuln_types, has_fix_count = {}, {}, 0
for r in deduped:
    sources[r["source"]] = sources.get(r["source"], 0) + 1
    vuln_types[r["vulnerability_type"]] = vuln_types.get(r["vulnerability_type"], 0) + 1
    if r["has_fix"]:
        has_fix_count += 1

stats = {
    "total_records":       len(deduped),
    "records_with_fix":    has_fix_count,
    "records_without_fix": len(deduped) - has_fix_count,
    "by_source":           sources,
    "top_vuln_types":      dict(sorted(vuln_types.items(), key=lambda x: -x[1])[:20]),
}

unified_path = OUT_DIR / "unified_dataset.jsonl"
print(f"\n=== Writing {unified_path} ===")
with open(unified_path, "w", encoding="utf-8") as f:
    for r in deduped:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")
print(f"  Written: {len(deduped)} records → {unified_path}")

chat_path = OUT_DIR / "train_chat_template.jsonl"
print(f"\n=== Writing {chat_path} ===")
chat_count = 0
with open(chat_path, "w", encoding="utf-8") as f:
    for r in deduped:
        ct = to_chat_template(r, include_explanation=INCLUDE_EXPLANATION)
        if ct:
            f.write(json.dumps(ct, ensure_ascii=False) + "\n")
            chat_count += 1
print(f"  Written: {chat_count} chat-template records → {chat_path}")

if WRITE_CSV and deduped:
    import csv as _csv
    csv_path = OUT_DIR / "unified_dataset.csv"
    print(f"\n=== Writing {csv_path} ===")
    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        writer = _csv.DictWriter(f, fieldnames=list(deduped[0].keys()))
        writer.writeheader()
        writer.writerows(deduped)
    print(f"  Written: {len(deduped)} rows → {csv_path}")

stats_path = OUT_DIR / "stats.json"
with open(stats_path, "w") as f:
    json.dump(stats, f, indent=2)

print(f"\n=== Stats ===")
print(json.dumps(stats, indent=2))
print(f"\n✓ Done! Output files:")
print(f"  {unified_path}")
print(f"  {chat_path}")
print(f"  {stats_path}")